## Chapter 02 

## RADONIKA LIBRARY

We start building the library now. We will have the standard structure. We will come back to this later once the library is prepared.

## 2.1 PROCESS LAYER

---

The file `process.py` is a 2-D stochastic process engine. Instead of reading it from line 1 to the end, we will build it bottom-up. Starting with a concrete process, finding what logic repeats, extracting that logic into base classes and helpers, and gradually expanding into more complex math. 

Every object in `process.py` models a 2-D trajectory $(X_t, Y_t)$. Rather than writing separate simulation, density, and plotting code for every single random process, we will make a file that has a separate definition for things that we will use again and again.

```text
Concrete Processes (normal, bm, poisson, stocks...)
├── Defined using their unique maths, how to step forward
│
└── Intermediate Wrappers (_IID)
    ├── Groups processes that are iid
    │
    └── Class (Process)
        ├── Provides the universal engine: paths(), sample(), pdf(), measure tilting, & UI plots
        │
        └── 4 tiny helpers (_pair, _vol, _bvn_pdf, _gauss_paths)
```

## 2.1.1 Two-Dimensional Gaussian Process

We will start with making the process $iid$ Normal in 2D. In general they are defined using mean and variance where both are 2D vectors because the process is 2D. We want to add a feature that instead of `normal(mean=[0.0, 2.0], sigma=[1.0, 0.5])` if we simply write `normal(mean=0.0, sigma=1.0)` it will be able to recognize it as a 2D array by repeating the values as `normal(mean=[0.0, 0.0], sigma=[1.0, 1.0])`. That is where the first helper comes ito picture.

In [1]:
def _pair(v, name="parameter"):
    a = np.asarray(v, float)
    if a.ndim == 0:
        return np.repeat(a, 2)
    if a.shape == (2,):
        return a.copy()
    raise ValueError(f"{name} must be a scalar (used for both coordinates) or a length-2 sequence")

When entry is already 2-element list, `return a.copy()` is simply saying not to change anything about the entry and returning a copy of the entry (a copy instead of the original one to avoid re-writings). everything else raises Error. Since we plan on making $iid$ Cauchy, Normal, Uniform etc, the main underlying engine of generating the array etc can be created separately under `_IID` and `_IID` will need `Process`.

In [3]:
class normal:
    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")                  #the _pair helper is used
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")

Technically it should be `normal(_IID):` where `_IID` will simulate the trajectories. That will further need `Process` as we said before. So, let us start there now, from making the `class Process`.

```text
shape = (n, len(times), 2)
         │      │        └─ the two coordinates, X and Y
         │      └────────── how many time points
         └───────────────── how many paths
```
If we had to do this with nothing else, the whole Process would be this:

In [9]:
class Process:
    def simulate(self, n, times, start, rng):
        raise NotImplementedError

`Process` itself has no idea how to simulate anything. It says “every process must have a `simulate`” and raises `NotImplementedError` if a subclass forgets. Each specific process (Cauchy, Brownian motion, and so on) fills in the actual recipe. We will now make all these small building blocks for the `class Process` such as `_rng`, `simulate`, `_start`, `paths` etc. We have already discussed what `rng` does in Python. Let us see what the others are doing one by one.

One rule is written in the docstring that `times[0]` must be 0, and the first column of the output must equal `start`. Every path starts at the starting point. The `simulate` also needs `start`, the point `(x0, y0)` where every path begins. The user might give it in a few different ways, or not at all, so the process needs one place that turns whatever they gave into a clean array.

In [10]:
def _start(self, start):
    s = self.default_start if start is None else start
    s = np.asarray(s, float)
    if s.ndim == 0:
        s = np.repeat(s, self.dim)
    if s.shape != (self.dim,):
        raise ValueError(f"start must have {self.dim} coordinates")
    return s

It does four small jobs, in order:

1. Fill in a default. If the user passed nothing (None), use `self.default_start`, which is (0.0, 0.0)` for most processes. Stock prices override it to `(1.0, 1.0)`, because a price starting at 0 would be stuck at 0 forever.
2. Convert to a NumPy array of floats. A tuple `(0, 0)` becomes array `([0., 0.])`.
3. Allow a shortcut. If the user passed a single number like 5, `s.ndim == 0` is true, and `np.repeat(s, 2)` turns it into `[5., 5.]`. So `start=5` means “start both coordinates at 5”.
4. Check the shape. After all that, the result must be exactly length 2. If not, it raises a clear error instead of failing mysteriously later.

We now have `_rng` and `_start`. We also have `simulate`. But calling `simulate` directly is clumsy because the user would have to build times, start and rng by hand.

`times = np.linspace(0, 1, 201)`

`X = process.simulate(20, times, process._start(None), process._rng(None))`

So `Process` gives a friendly method that does all of that named `paths`.

In [13]:
def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
    times = np.linspace(0.0, T, steps + 1)
    return times, self.simulate(n, times, self._start(start), self._rng(seed))

Note that `paths` is using `simulate` that further has `_rng` and `_start` that we built before. Every argument has a default, so `process.paths()` alone already works giving 20 paths, on $[0, 1]$, 200 steps, starting at the origin. So far we have:  

In [28]:
class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))    

`paths()` gives you whole trajectories. Often you want a different question: “at time t = 3, what values does X_t take?” To see that distribution, you need many independent draws of (X_3, Y_3), with no need for the path leading up to it. This is the law of the process at that time (similar to the 2D case). The simplest way is to simulate paths up to time t and keep only the last time point. For that we need `sample` and `sample_slices`. The cloud of points is a picture of the law. It’s dense where the process likely lands and thin where it rarely does and that is what `sample` will do. 

Where does `sample_slices` come into the picture? We will make it like an animation that shows the law evolving over time. For that we need a process to simulate the cloud at several time points and that is what `sample_slice` will do. `sample_slice` can create many clouds over many time points and `sample` just looks at them one by one.

In [29]:
def sample_slices(self, times, n, rng, start):
    X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
    return np.transpose(X[:, 1:, :], (1, 0, 2))

Recall, $X[3, 10, 1]$ means “path number 3, at the 10th time point, the Y coordinate”. So, `X[:, 1:, :]` means from the time array we are dropping the initial slice, start that array from 1. This is because we had `np.r_` that added $0$ in begining of `times`. So, something like $[1, 2, 3]$ became $[0, 1, 2, 3]$ but now we have to remove the initial time slice. The $0$ was added since that is how simulate works and the form of inputs are predefined. Once that simulation is done, we do need to return those time slices only that the user asked. The transpose here exchanges the first two component of $X$[path number, times, coordiate number] to $X$[times, path number, coordiate number]. This will read "(slice, n, 2)" which is used in next part.

In [31]:
def sample_slices(self, times, n, rng, start):
    X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
    return np.transpose(X[:, 1:, :], (1, 0, 2))
    
def sample(self, t, n=10000, start=None, seed=None):
    return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

The `[0]` at the end of the def of `sample` is simply because sample took only one time as entry, namely, `[t]` and then we want to look at them one by one. So, `[0]` says move along the $0^{th}$ coordinate (time) one by one; so basically instead of stacks of clouds, it is just one cloud at a time. Now we have the updated class as:

In [32]:
class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))   

    def sample_slices(self, times, n, rng, start):
        X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
        return np.transpose(X[:, 1:, :], (1, 0, 2))
    
    def sample(self, t, n=10000, start=None, seed=None):
        return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

There will be other things to add here, we will come to them in due time. For now, let us move on to another application of `simulate` to generate an actual process (which was kind of the main task and where we started this discussion). 

Say we want to create $iid$ Cauchy. Usinf the built process will give us all those properties to out cauchy process. So, doing it by hand using only `Process` looks like:

In [34]:
class cauchy(Process):
    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start                                    # time 0 = start point
        out[:, 1:, :] = rng.standard_cauchy((n, len(times)-1, 2))   # fresh noise afterwards
        return out

This works. Now if we want for Uniform we redo this:

In [35]:
class cauchy(Process):
    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start                                    # time 0 = start point
        out[:, 1:, :] = rng.uniform((n, len(times)-1, 2))   # fresh noise afterwards
        return out

Now we can do this for all processes we want such as exponential, Laplace, Student-t, lognormal, etc. This is a signal to make a shared engine for $iid$ processes. So we make another `class` named `_IID(Process)` and it has `Process` as argument since we do not want to miss out on the properties we built so far. 

In [37]:
class _IID(Process):
    """X_t independent across time; coordinates independent scipy distributions."""

    def __init__(self, frozen, seed=None, desc="iid"):
        super().__init__(seed)
        self.frozen = list(frozen)
        self._desc = desc

    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start
        for k in range(2):
            out[:, 1:, k] = self.frozen[k].rvs(size=(n, len(times) - 1), random_state=rng)
        return out

This `simulate` under `class _IID(Process)` is exactly same as the chunk of codes we were repeating again and again with two changes. 
1. `self.frozen[k].rvs(...)` replaces the hard-coded `rng.standard_cauchy`. It asks whatever distribution is stored for random draws, which is the whole point.
2. A loop over `k` in `range(2)`, because `self.frozen` is a list of two distributions, one for X (k=0) and one for Y (k=1). That lets X and Y have different parameters, for example X ~ Normal(0, 1) and Y ~ Normal(5, 2). Each fills its own slice `out[:, 1:, k]`. This gives us another degree of freedom hence.